# Node sets — faces, grain boundary, and grains

After ELSETs:

- `build_boundary_nsets()` → `LEFT` `RIGHT` `TOP` `BOTTOM` and corners
- `build_gb_nset()` → `GB`
- `build_grain_nsets()` → keys matching ELSET names (`grain.1`, …)

`plot_by_grain(..., show_nsets=True)` marks the four faces. Abaqus INP writes
`NS_LEFT` / `RIGHT` / `TOP` / `BOTTOM` / `GB` (grain NSETs stay on the mesher
object; they are not yet keyword-exported).

Requires `gmsh` (`pip install upxo[mesh]`). `confMesh2d` (pygmsh) is deprecated.
Canonical mesh-only demo: `confMesh2d_gmsh.ipynb`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import box

from upxo.meshing.gsmesh2d import mesh_gs
from upxo.meshing.writer_ABQ import summarize_inp


In [ ]:
cells = {
    1: box(0, 0, 2, 2),
    2: box(2, 0, 4, 2),
    3: box(0, 2, 2, 4),
    4: box(2, 2, 4, 4),
}

In [ ]:
result = mesh_gs(cells, mesh_size_gb=0.35, mesh_size_bulk=0.7,
                 mesh_algo=6, recombine_to_quads=False)
m = result['mesher']
m.form_elsets_gmsh()
m.build_boundary_nsets()
m.build_gb_nset()
m.build_grain_nsets()
for k, ids in m.nsets.items():
    print(f'{k:16s} {len(ids):5d}')

In [ ]:
fig, ax = m.plot_by_grain(figsize=(6, 6), show_gb=True, show_nsets=True,
                          title='face NSETs + GB overlay')
g1 = m.nsets['grain.1']
ax.plot(m.nodes[g1, 0], m.nodes[g1, 1], 'k.', ms=3, label='grain.1 nodes')
ax.legend(loc='upper right', fontsize=8)
fig

In [ ]:
out = Path.cwd() / 'confMesh2d_nsets_out'
inp = m.export_abaqus_inp(out / 'rve_cps3.inp', plane='stress')
text = Path(inp).read_text(encoding='utf-8')
nset_lines = [ln for ln in text.splitlines() if ln.startswith('*Nset')]
inp, summarize_inp(inp), nset_lines